# QuantAI specialist model — v4 training and evaluation

This notebook fine-tunes an open-weight instruction model (Qwen3-4B-Instruct) for QuantAI with LoRA/QLoRA. It does **not** call a paid LLM API. The result is a specialised fine-tune of a pretrained model, not a model trained from scratch.

**What v4 changes:** the training and evaluation examples now contain the *exact* context QuantAI's browser sends (dataset variables, data checks, earlier analyses, chat history and retrieved knowledge), built by the same code the local API uses. Every expected action was run in QuantAI's real engine before it was kept. v3 was trained on a short `Context: variables are …` format that the app never sends.

**Before you start:** choose Runtime → Change runtime type → Hardware accelerator → GPU, then reconnect. If Colab does not provide a GPU, this notebook still runs CPU-safe data validation and tests, but clearly skips model training, GPU evaluation and export. GPU availability depends on Colab capacity and account limits. Runtime estimates are approximate; checkpoints are saved to Google Drive when enabled.

No Hugging Face token is needed. Do not add secrets or private research data to this notebook.


In [ ]:
# Detect the runtime first, but do not abort CPU-safe validation when no GPU is attached.
import os, sys, subprocess, pathlib
try:
    import torch
except ImportError as exc:
    raise RuntimeError("PyTorch is missing from this Colab runtime. Select a standard Colab Python runtime and reconnect.") from exc

CUDA_AVAILABLE = bool(torch.cuda.is_available())
print("CUDA available:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
else:
    print(
        "GPU training is unavailable in this session. CPU-only data validation and unit tests can still run. "
        "To train, open Runtime > Change runtime type > Hardware accelerator > GPU, save, reconnect, "
        "then run this notebook again. Colab availability depends on your account and current capacity."
    )

REPO = "/content/model-analysis--hub"
BRANCH = "quantai-action-completeness-v2"
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "https://github.com/kamaldeenjnr/model-analysis--hub.git", REPO],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)

os.chdir(os.path.join(REPO, "quantai-model"))
print("Working directory:", os.getcwd())
print("Repository revision:", subprocess.check_output(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
# Deliberately do not uninstall Colab's preinstalled packages.
# Removing Gradio/diffusers/torchao can create dependency conflicts and is not needed by QuantAI.
print("Keeping the Colab base environment intact.")

In [ ]:
# Install the project dependencies. Colab's existing CUDA-enabled PyTorch is retained
# whenever it already satisfies requirements.txt.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
if pip_check.returncode == 0:
    print("pip dependency consistency check passed.")
else:
    print("WARNING: pip reports environment conflicts. These may involve unrelated Colab preinstalled packages.")
    print(pip_check.stdout or pip_check.stderr)
    print("Continue only if the QuantAI imports and preflight tests below pass; do not ignore an import/test failure.")

# Re-import after installation and re-check CUDA in case the runtime changed.
import torch
CUDA_AVAILABLE = bool(torch.cuda.is_available())
print("CUDA available after dependency installation:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print("GPU:", torch.cuda.get_device_name(0))

## 0. Keep outputs on Google Drive (recommended)

With `PERSIST_TO_DRIVE = True`, the `outputs` folder lives in `MyDrive/QuantAI/outputs`, so checkpoints, evaluation reports and exported models survive a disconnect. Your v3 files already there are kept: v4 uses its own names. After a disconnect, reconnect and **Run all**: training resumes from the newest checkpoint.


In [ ]:
PERSIST_TO_DRIVE = True  # set False to keep outputs only for this Colab session
import shutil
OUT = pathlib.Path('outputs')
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    target = pathlib.Path('/content/drive/MyDrive/QuantAI/outputs')
    target.mkdir(parents=True, exist_ok=True)
    if OUT.is_symlink():
        OUT.unlink()
    if OUT.exists():
        # Outputs from earlier in this session: copy them to Drive, keep the local copy aside.
        shutil.copytree(OUT, target, dirs_exist_ok=True)
        OUT.rename('outputs_local_backup')
        print('Copied existing local outputs to Drive; the local copy is in outputs_local_backup/')
    OUT.symlink_to(target, target_is_directory=True)
    print('outputs ->', target)
    print('Already in Drive:', sorted(x.name for x in target.iterdir()) or 'nothing yet')
else:
    OUT.mkdir(exist_ok=True)
    print('Outputs are kept only for this session:', OUT.resolve())


## 1. Validate the data and code

Checks the v4 files (schema, every action against the contract, variables visible in the context, no held-out scenario in training, no near-duplicate wording) and runs the offline tests. These checks do not show that a model is good; the evaluation in section 3 does that.


In [ ]:
# Run each preflight command with check=True so a failure is shown at the exact step.
commands = [
    [sys.executable, "validate_data.py", "--train", "data/v4/train.jsonl", "--eval", "data/v4/eval.jsonl"],
    [sys.executable, "datagen/audit.py", "data/v4/train.jsonl", "data/v4/eval.jsonl"],
    [sys.executable, "-m", "pytest", "-q", "test_pipeline_v4.py", "test_local_service.py", "test_frontend_contract.py"],
]
for command in commands:
    print("\n>>>", " ".join(command), flush=True)
    subprocess.run(command, check=True)
print("\nAll data, engine-audit and offline test checks passed.")

## 2. Train the v4 adapter

Choose the run size:
- **Standard** (`TRAIN_ROWS = 650`): an even sample of every kind of request, about 3 hours on a T4. Fits a free Colab session.
- **Full** (`TRAIN_ROWS = 0`): all 1,124 training rows, about 5 hours. Use it if you have a longer session (or resume after a disconnect).

One epoch is used because each row is a different request; the earlier v3 run needed three epochs over only 93 rows. A checkpoint is written every 10 steps. If no GPU is attached, training is skipped without aborting the CPU-safe validation workflow. The first lines of output report the real token lengths; training stops (instead of silently cutting text) if any row is longer than `--max-length`.


In [ ]:
TRAIN_ROWS = 650   # Use 650 examples for the standard run; use 0 for all v4 training examples.
TRAIN_OUTPUT = pathlib.Path("outputs/quantai-qwen3-4b-lora-v4")

if not CUDA_AVAILABLE:
    print(
        "Training was skipped because this Colab session does not have an available GPU. "
        "No training has started. Select Runtime > Change runtime type > GPU, reconnect, "
        "and run the notebook again."
    )
else:
    import platform
    print("Python version:", platform.python_version())
    print("PyTorch version:", torch.__version__)
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

    try:
        import bitsandbytes
        print("bitsandbytes version:", bitsandbytes.__version__)
    except Exception as exc:
        raise RuntimeError(
            "The GPU is available, but bitsandbytes could not be loaded. "
            "Training has not started. Check the CUDA, PyTorch and bitsandbytes compatibility "
            "before trying again."
        ) from exc

    command = [
        sys.executable, "train_lora.py",
        "--train", "data/v4/train.jsonl",
        "--eval", "data/v4/eval.jsonl",
        "--output", str(TRAIN_OUTPUT),
        "--epochs", "1",
        "--max-train-rows", str(TRAIN_ROWS),
        "--resume",
    ]
    print("\nStarting QuantAI v4 training:")
    print(" ".join(command), flush=True)

    # Capture the complete training output so the real error is visible if training fails.
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    if result.stdout:
        print(result.stdout, flush=True)

    if result.returncode != 0:
        raise RuntimeError(
            "QuantAI v4 training failed with exit code "
            + str(result.returncode)
            + ". Read the full training error printed above. "
              "This message replaces the less informative CalledProcessError."
        )

    manifest = TRAIN_OUTPUT / "training_manifest.json"
    adapter = TRAIN_OUTPUT / "adapter_config.json"
    if not manifest.exists() or not adapter.exists():
        raise RuntimeError(
            "The training process ended, but the adapter or training manifest is missing. "
            "Do not continue to evaluation."
        )
    print("Training completed. The adapter and training manifest were found.")

## 3. Evaluate base, v3 and v4 on the same held-out cases

All models answer the same 236 held-out cases (research scenarios never used for training, worded differently), loaded the same way (4-bit on the GPU):
- **Base**: untouched Qwen3-4B with the v4 instructions — what prompting alone achieves.
- **v3**: your previous adapter, with the v3 prompt it was trained on — how v3 would behave inside the real app.
- **v4**: the new adapter.

Each run takes roughly 10–20 minutes. *Lenient match* accepts only differences that change nothing in the engine (predictor order, `auto` versus the model the engine would pick, swapped variables in a correlation). Field scores show which part of an action goes wrong.


In [ ]:
V4_ADAPTER = pathlib.Path("outputs/quantai-qwen3-4b-lora-v4")
V3_ADAPTER = pathlib.Path("outputs/quantai-qwen3-4b-lora-v3")

if not CUDA_AVAILABLE:
    print("EVALUATION SKIPPED: GPU unavailable. Evaluation of the 4B model is not attempted on CPU in this notebook.")
elif not (V4_ADAPTER / "adapter_config.json").exists():
    print("EVALUATION SKIPPED: the v4 adapter is not present. Complete a successful training run first.")
else:
    eval_commands = [
        [sys.executable, "evaluate.py", "--model", str(V4_ADAPTER), "--cases", "data/v4/eval.jsonl", "--report", "outputs/v4-evaluation.json", "--batch-size", "1"],
        [sys.executable, "evaluate.py", "--baseline-only", "--cases", "data/v4/eval.jsonl", "--report", "outputs/base-evaluation-v4set.json", "--batch-size", "1"],
    ]
    if (V3_ADAPTER / "adapter_config.json").exists():
        eval_commands.append([
            sys.executable, "evaluate.py", "--model", str(V3_ADAPTER), "--prompt-version", "v3",
            "--cases", "data/v4/eval.jsonl", "--report", "outputs/v3-evaluation-v4set.json", "--batch-size", "1"
        ])
    else:
        print("v3 comparison skipped: previous v3 adapter not found in the Drive outputs folder.")

    for command in eval_commands:
        print("\n>>>", " ".join(command), flush=True)
        subprocess.run(command, check=True)

    required_reports = ["outputs/v4-evaluation.json", "outputs/base-evaluation-v4set.json"]
    missing = [name for name in required_reports if not pathlib.Path(name).exists()]
    if missing:
        raise RuntimeError("Evaluation did not create required report(s): " + ", ".join(missing))
    print("v4 and baseline evaluation reports verified.")

In [ ]:
import json
report_paths = {
    "Base": pathlib.Path("outputs/base-evaluation-v4set.json"),
    "v3": pathlib.Path("outputs/v3-evaluation-v4set.json"),
    "v4": pathlib.Path("outputs/v4-evaluation.json"),
}
reports = {name: json.loads(path.read_text()) for name, path in report_paths.items() if path.exists()}
if not reports:
    print("No evaluation reports are available yet. Run this section after GPU training and evaluation.")
else:
    rows = [
        ("Valid JSON", "valid_json_rate"),
        ("Action contract valid", "action_contract_valid_rate"),
        ("Action type correct", "action_type_accuracy"),
        ("Exact action match", "exact_action_match_rate"),
        ("Lenient action match", "lenient_action_match_rate"),
        ("Declines/asks when it should", "null_when_expected_rate"),
        ("Declines when an action was wanted", "null_when_action_expected_rate"),
        ("Replies with numbers not in context", "replies_with_numbers_not_in_context_rate"),
    ]
    print(f"{'Metric':38}" + "".join(f"{name:>10}" for name in reports))
    for label, key in rows:
        values = []
        for report in reports.values():
            value = report.get(key)
            values.append(f"{value:10.1%}" if isinstance(value, (int, float)) else f"{'n/a':>10}")
        print(f"{label:38}" + "".join(values))

    v4 = reports.get("v4")
    if v4:
        interval = v4.get("lenient_action_match_rate_ci95")
        if interval and len(interval) == 2:
            print(f"\nv4 lenient match 95% CI: {interval[0]:.1%} to {interval[1]:.1%} (n = {v4.get('cases', 0)})")
        print("\nField accuracy (v4):")
        for field, score in v4.get("field_scores", {}).items():
            if not field.endswith("precision"):
                print(f"  {field:34} {score['mean']:6.1%}  (n={score['n']})")
        print("\nLenient match by request type:")
        names = list(reports)
        print(f"  {'intent':22}" + "".join(f"{name:>8}" for name in names) + "     n")
        for intent, detail in v4.get("by_intent", {}).items():
            scores = "".join(f"{reports[name].get('by_intent', {}).get(intent, {}).get('lenient_match', float('nan')):8.0%}" for name in names)
            print(f"  {intent:22}{scores}  {detail.get('n', 0):4}")
        print("\nv4 cases that did not match (first 25):")
        misses = [item for item in v4.get("results", []) if not item.get("lenient_action_match")]
        for item in misses[:25]:
            got = (item.get("parsed") or {}).get("action") if isinstance(item.get("parsed"), dict) else item.get("raw_output", "")[:120]
            print(f"\n[{item.get('intent')}] {item.get('message', '')[:140]}\n  expected: {json.dumps(item.get('expected', {}).get('action'))[:220]}\n  got:      {json.dumps(got)[:220]}")
        print(f"\n{len(misses)} of {len(v4.get('results', []))} v4 cases did not match.")

## 4. Export a standalone v4 model

Merges the adapter into the base weights (about 8 GB) and records the prompt version, so `serve.py` automatically uses the v4 prompt with this model. Check the base model's licence before redistributing weights.

Optional: the scores above were measured in 4-bit. To confirm the exported fp16 model behaves the same, run the commented line (about 20 minutes).


In [ ]:
MERGED_OUTPUT = pathlib.Path("outputs/quantai-merged-model-v4")
if not CUDA_AVAILABLE:
    print("EXPORT SKIPPED: a GPU is not available in this session.")
elif not (pathlib.Path("outputs/quantai-qwen3-4b-lora-v4") / "adapter_config.json").exists():
    print("EXPORT SKIPPED: the v4 adapter is missing. Complete training first.")
else:
    command = [
        sys.executable, "export_model.py",
        "--adapter", "outputs/quantai-qwen3-4b-lora-v4",
        "--output", str(MERGED_OUTPUT),
    ]
    print("\n>>>", " ".join(command), flush=True)
    subprocess.run(command, check=True)
    required = ["config.json", "quantai_prompt.json", "tokenizer_config.json"]
    missing = [name for name in required if not (MERGED_OUTPUT / name).exists()]
    if missing:
        raise RuntimeError("Model export is incomplete; missing: " + ", ".join(missing))
    print("Export verified:", MERGED_OUTPUT.resolve())
    # Optional: compare the merged model with the adapter using the same held-out cases.
    # subprocess.run([sys.executable, "evaluate.py", "--model", str(MERGED_OUTPUT), "--precision", "float16",
    #                 "--cases", "data/v4/eval.jsonl", "--report", "outputs/v4-merged-fp16-evaluation.json"], check=True)

## 5. Check what was saved


In [ ]:
for name in [
    "quantai-qwen3-4b-lora-v4", "v4-evaluation.json", "base-evaluation-v4set.json",
    "v3-evaluation-v4set.json", "quantai-merged-model-v4"
]:
    path = pathlib.Path("outputs") / name
    print(f"{name:32} {'present' if path.exists() else 'not created'}")
    if path.is_dir():
        size = sum(item.stat().st_size for item in path.rglob("*") if item.is_file())
        print(f"  size: {size / (1024**3):.2f} GiB")
print("\nRuntime status:", "GPU available" if CUDA_AVAILABLE else "CPU only — training/evaluation/export were skipped")

## Release checklist

- [ ] Read the v4 failure list above and decide which failures matter most.
- [ ] Spot-check v4 on a few of your own real questions in the app before relying on it.
- [ ] Have a second person review a sample of training and held-out examples.
- [ ] Serve v4 locally (`QUANTAI_MODEL_PATH=outputs/quantai-merged-model-v4`) and check latency and memory on the target machine.
- [ ] Keep statistical calculations in the deterministic engine and validate every requested action.
- [ ] Do not claim production readiness until independent review and acceptance thresholds pass.
